# JED Attack — Agent Security | DGX Spark GB10

Mirrors `kaggle_notebook.ipynb` but runs on DGX Spark GB10 using **llama-server** (llama.cpp)
serving the competition GGUF models locally at `http://localhost:8082/v1`.

## Prerequisites — run these in tmux BEFORE running any cells

```zsh
# GPT-OSS 20B:
tmux new -s llama
MODEL=gpt_oss bash scripts/llama-serve.sh   # port 8082
# Ctrl+B D to detach

# Gemma 4 26B (after GPT-OSS run — stop llama tmux session first):
MODEL=gemma bash scripts/llama-serve.sh
```

## Cell execution order

| Cell | Purpose |
|---|---|
| 1 | Setup: venv, imports, load attack |
| 2 | Health-check llama-server |
| 3 | Run GPT-OSS 20B (9000s budget) |
| 4 | Run Gemma 4 26B (9000s budget) |
| 5 | Combined score summary |

Cells 3 and 4 run sequentially (one model at a time). Switch the llama-server between runs.

## 1. Setup

In [ ]:
import importlib.util, json, os, sys, time
from pathlib import Path

REPO_DIR = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').parent.resolve()
# Fallback: set REPO_DIR manually if auto-detection fails
# REPO_DIR = Path('/home/msusol/LosusAI/Projects/Kaggle/kaggle-ai-agent-security-multi-step-tool-attacks/jed-redteam-attack')
REPO_DIR = Path('/home/msusol/LosusAI/Projects/Kaggle/kaggle-ai-agent-security-multi-step-tool-attacks/jed-redteam-attack')

VENV_PYTHON = Path.home() / 'LosusAI/Projects/Kaggle/.venv/bin/python'
ARTIFACTS   = REPO_DIR / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Server config (llama-server via scripts/llama-serve.sh)
SERVER_PORT = int(os.environ.get('PORT', 8082))
SERVER_BASE = f'http://localhost:{SERVER_PORT}/v1'
BUDGET_S    = int(os.environ.get('BUDGET', 9000))
SEED        = 123

print(f'REPO_DIR   : {REPO_DIR}')
print(f'server     : {SERVER_BASE}')
print(f'budget     : {BUDGET_S}s')
print(f'seed       : {SEED}')
print(f'artifacts  : {ARTIFACTS}')

# Load attack
from local_harness import load_attack_class, evaluate_redteam_local, make_env
from aicomp_sdk.attacks import AttackRunConfig

AttackAlgorithm = load_attack_class(str(REPO_DIR / 'attack.py'))
print(f'\nAttackAlgorithm: {AttackAlgorithm}')

## 2. Health-check llama-server

In [ ]:
import urllib.request, json as _json

def check_server(base_url: str) -> dict | None:
    try:
        with urllib.request.urlopen(f'{base_url}/models', timeout=5) as r:
            data = _json.loads(r.read())
        models = [m['id'] for m in data.get('data', [])]
        print(f'✓ llama-server UP at {base_url}')
        print(f'  models: {models}')
        return data
    except Exception as e:
        print(f'✗ llama-server NOT reachable at {base_url}')
        print(f'  error: {e}')
        print(f'  → start with: MODEL=<gpt_oss|gemma> bash scripts/llama-serve.sh')
        return None

server_info = check_server(SERVER_BASE)

## 3. Run against GPT-OSS 20B

> **Before running this cell:** ensure llama-server is loaded with `MODEL=gpt_oss`
>
> ```zsh
> MODEL=gpt_oss bash scripts/llama-serve.sh
> ```

In [ ]:
MODEL_NAME = 'gpt_oss'

os.environ['VLLM_BASE_URL'] = SERVER_BASE
os.environ['VLLM_MODEL']    = MODEL_NAME

print(f'=== GPT-OSS 20B | budget={BUDGET_S}s | server={SERVER_BASE} ===')
assert check_server(SERVER_BASE) is not None, 'Start llama-server first (MODEL=gpt_oss)'

config  = AttackRunConfig(time_budget_s=BUDGET_S, max_tool_hops=8, seed=SEED)
env     = make_env(use_llm=True, seed=SEED, verbose=False)

t0      = time.time()
result  = evaluate_redteam_local(AttackAlgorithm, env, config, verbose=True)
wall_s  = round(time.time() - t0, 1)

gpt_summary = {
    'model'                      : MODEL_NAME,
    'score_normalized_0_to_1000' : result.score,
    'score_raw'                  : result.score_raw,
    'findings_count'             : result.findings_count,
    'unique_cells'               : result.unique_cells,
    'evaluation_time_s'          : round(result.time_taken, 1),
    'wall_time_s'                : wall_s,
}
(ARTIFACTS / f'{MODEL_NAME}_summary.json').write_text(json.dumps(gpt_summary, indent=2))
print(f'\nGPT-OSS summary:')
print(json.dumps(gpt_summary, indent=2))

## 4. Run against Gemma 4 26B

> **Before running this cell:** stop the GPT-OSS llama-server and reload with `MODEL=gemma`
>
> ```zsh
> # In the llama tmux session:
> # Ctrl+C to stop, then:
> MODEL=gemma bash scripts/llama-serve.sh
> ```

In [ ]:
MODEL_NAME = 'gemma'

os.environ['VLLM_BASE_URL'] = SERVER_BASE
os.environ['VLLM_MODEL']    = MODEL_NAME

print(f'=== Gemma 4 26B | budget={BUDGET_S}s | server={SERVER_BASE} ===')
assert check_server(SERVER_BASE) is not None, 'Start llama-server first (MODEL=gemma)'

config  = AttackRunConfig(time_budget_s=BUDGET_S, max_tool_hops=8, seed=SEED)
env     = make_env(use_llm=True, seed=SEED, verbose=False)

t0      = time.time()
result  = evaluate_redteam_local(AttackAlgorithm, env, config, verbose=True)
wall_s  = round(time.time() - t0, 1)

gemma_summary = {
    'model'                      : MODEL_NAME,
    'score_normalized_0_to_1000' : result.score,
    'score_raw'                  : result.score_raw,
    'findings_count'             : result.findings_count,
    'unique_cells'               : result.unique_cells,
    'evaluation_time_s'          : round(result.time_taken, 1),
    'wall_time_s'                : wall_s,
}
(ARTIFACTS / f'{MODEL_NAME}_summary.json').write_text(json.dumps(gemma_summary, indent=2))
print(f'\nGemma summary:')
print(json.dumps(gemma_summary, indent=2))

## 5. Combined scores

In [ ]:
# Load from saved artifacts if cells 3/4 were not run in this session
def _load(name):
    p = ARTIFACTS / f'{name}_summary.json'
    if p.exists():
        return json.loads(p.read_text())
    return None

gpt   = locals().get('gpt_summary')   or _load('gpt_oss')
gemma = locals().get('gemma_summary') or _load('gemma')

print('=' * 60)
print('  DGX SPARK GB10 — COMBINED SCORES')
print('  (mirrors Kaggle Cell 6/7 output format)')
print('=' * 60)

for label, s in [('GPT-OSS 20B', gpt), ('Gemma 4 26B', gemma)]:
    if s:
        print(f'\n{label}:')
        print(f'  score_normalized : {s["score_normalized_0_to_1000"]}')
        print(f'  score_raw        : {s["score_raw"]}')
        print(f'  findings         : {s["findings_count"]}')
        print(f'  unique_cells     : {s["unique_cells"]}')
        print(f'  eval_time        : {s["evaluation_time_s"]}s')
    else:
        print(f'\n{label}: not yet run')

if gpt and gemma:
    mean = (gpt['score_normalized_0_to_1000'] + gemma['score_normalized_0_to_1000']) / 2
    print(f'\n  local_public_mean: {mean:.4f}')
    print(f'  (Kaggle public score = mean of both models)')

print()